Import Libraries

In [231]:
import gymnasium as gym
import numpy as np
import random
import ale_py
from gymnasium.envs.registration import make, pprint_registry, register, registry, spec
from IPython.display import clear_output
import time

Initialize Gym Variables

In [258]:
# gym.register_envs(ale_py)
# env_name = "ALE/Pong-v5" # Wont render in ipynb

# env_name = "MountainCar-v0" # Example Discrete
# env_name = "MountainCarContinuous-v0" # Example Continuous


# env_name = "FrozenLake-v1"
try:
    register(
        id="FrozenLakeNoSlip-v1",
        entry_point="gymnasium.envs.toy_text.frozen_lake:FrozenLakeEnv",
        kwargs={"map_name": "4x4", "is_slippery": False},
        max_episode_steps=100,
        reward_threshold=0.78  # optimum = 0.74
    )
except:
    pass
env_name = "FrozenLakeNoSlip-v1"

render_mode_name = "human"
env = gym.make(env_name, render_mode = render_mode_name)


print("Observation Space: ", env.observation_space)
print("Action Space: ", env.action_space, " as type ", type(env.action_space))

Observation Space:  Discrete(16)
Action Space:  Discrete(4)  as type  <class 'gymnasium.spaces.discrete.Discrete'>


c:\Users\Reece S\Documents\GitHub\capstone4ds\.dqn_pong_env\Lib\site-packages\gymnasium\envs\registration.py:636: UserWarning: WARN: Overriding environment FrozenLakeNoSlip-v1 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


Define Agent

In [233]:
class Agent():
    def __init__(self, env):
        self.is_discrete = type(env.action_space) == gym.spaces.discrete.Discrete
        print("Is Discrete? ", self.is_discrete)

        if self.is_discrete:
            self.action_size = env.action_space.n
            print("Action size:", self.action_size)
        else:
            self.action_space_low = env.action_space.low
            self.action_space_high = env.action_space.high
            self.action_shape = env.action_space.shape
            print("Action range:", self.action_space_low, self.action_space_high)
    
    
    def get_action(self, observation):
        if self.is_discrete:
            action = random.choice(range(self.action_size))
        else:
            action = np.random.uniform(self.action_space_low, self.action_space_high, self.action_shape)

        return action


Define QAgent

In [253]:
class QAgent(Agent):
    def __init__(self, env, discount_rate = 0.97, learning_rate=0.01, epsilon = 1.0):
        super().__init__(env)
        self.observation_size = env.observation_space.n
        print("Observation size:", self.observation_size)

        self.eps = epsilon
        self.learning_rate = learning_rate
        self.discount_rate = discount_rate
        self.build_model()

    def build_model(self):
        self.q_table = 1e-4*np.random.random([self.observation_size, self.action_size])

    def get_action(self, observation):
        q_observation = self.q_table[observation]
        action_greedy = np.argmax(q_observation)
        action_random = super().get_action(observation)
        return action_random if random.random() < self.eps else action_greedy
    
    def train(self, experience):
        observation, action, next_observation, reward, done = experience
        print(experience)

        # Building Q function
        q_next = self.q_table[next_observation]
        q_next = np.zeros([self.action_size]) if done else q_next
        q_target = reward + self.discount_rate * np.max(q_next)

        q_update = q_target - self.q_table[observation,action]
        self.q_table[observation,action] += self.learning_rate * q_update

        if done:
            self.eps = self.eps * 0.99

agent = QAgent(env)


Is Discrete?  True
Action size: 4
Observation size: 16


Training Session

In [259]:

total_reward = 0

for ep in range(100):
    observation, info = env.reset()
    done = False
    while not done:
        action = agent.get_action(observation)
        next_observation, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        agent.train((observation, action, next_observation, reward, done))
        observation = next_observation
        total_reward += reward
        print("s:", observation, "a:", action)
        print("Episode: {}, Total reward: {}, Eps: {}".format(ep, total_reward, agent.eps))

        env.render()
        print(agent.q_table)
    
        time.sleep(0.05)
        clear_output(wait=True)

(14, np.int64(2), 15, 1, True)
s: 15 a: 2
Episode: 99, Total reward: 98, Eps: 0.006570483042414605
[[8.18948372e-05 6.27253303e-05 7.58344149e-02 7.26196219e-05]
 [4.32173432e-04 4.33742121e-06 1.67186490e-01 7.77393634e-05]
 [6.82454262e-05 3.22553068e-01 7.28739037e-05 5.07562268e-05]
 [5.52921955e-05 6.33829565e-05 3.06509474e-05 7.69755749e-05]
 [4.11757073e-05 6.59970776e-05 8.01362028e-05 8.47215449e-05]
 [5.08472946e-06 4.70372830e-05 5.88083388e-05 6.41161989e-05]
 [6.63108938e-05 5.39896590e-01 4.43660314e-05 2.76220468e-03]
 [3.34464142e-07 4.48872754e-05 7.10717976e-05 6.72078802e-05]
 [6.30434440e-05 5.42605264e-05 2.47967290e-05 2.76688510e-05]
 [4.76574680e-05 1.53485974e-03 1.85167432e-05 8.35756553e-05]
 [1.34919381e-04 7.74860552e-01 4.85823211e-05 2.59222883e-04]
 [4.01757661e-05 3.83423731e-05 1.48280811e-05 5.83743216e-05]
 [5.20205185e-05 7.84250338e-05 8.06296922e-05 6.44077863e-05]
 [5.60544560e-05 1.41753841e-05 4.98487897e-02 1.51291704e-05]
 [1.09110486e-03 2.